In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

# reload and reclean the data
df = pd.read_csv('/Users/kultumlhabaik/Documents/data-mining-final-project/data/raw/spotify_songs_JA.csv')
df = df.dropna(subset=['track_artist', 'track_name', 'track_album_name'])
df['release_year'] = df['track_album_release_date'].str[:4].pipe(pd.to_numeric, errors='coerce')
df_1990 = df[df['release_year'] >= 1990].copy()
df_1990 = df_1990[df_1990['track_popularity'] > 0].copy()

print('Rows loaded:', len(df_1990))


Rows loaded: 27791


The same song can appear multiple times in our dataset because it was pulled from multiple playlists. Removing these makes sure the song doesnt get counted twice while computing averages.

In [9]:
#Remove duplicated rows
df_deduplicated= df_1990.drop_duplicates(subset='track_id').copy()

print('Rows before:', len(df_1990))
print('Rows after: ', len(df_deduplicated))
print('Duplicate rows removed:   ', len(df_1990) - len(df_deduplicated))

Rows before: 27791
Rows after:  23831
Duplicate rows removed:    3960


In [10]:
# Filter artist with 3+ active years
active_years = df_deduplicated.groupby('track_artist')['release_year'].nunique()
artists_3_years = active_years[active_years >= 3].index.tolist()

df_qualified = df_deduplicated[df_deduplicated['track_artist'].isin(artists_3_years)].copy()

print('Total artists before filter:', df_deduplicated['track_artist'].nunique())
print('Artists with years>=3:', len(artists_3_years))
print('Rows before filter:', len(df_deduplicated))
print('Rows after filter:', len(df_qualified))


Total artists before filter: 9655
Artists with years>=3: 1094
Rows before filter: 23831
Rows after filter: 10196


In [15]:
# BUILD ARTIST YEAR PROFILES
# one track row per artist per year

artist_year = (
    df_qualified.groupby(['track_artist', 'release_year']).agg(
        mean_popularity   = ('track_popularity', 'mean'),
        median_popularity = ('track_popularity', 'median'),
        max_popularity    = ('track_popularity', 'max'),
        track_count       = ('track_popularity', 'count'),
        mean_danceability = ('danceability', 'mean'),
        mean_energy       = ('energy', 'mean'),
        mean_valence      = ('valence', 'mean'),
        mean_tempo        = ('tempo', 'mean'),
        mean_acousticness = ('acousticness', 'mean'),
        mean_speechiness  = ('speechiness', 'mean')
    )
    .reset_index()
    .round(4)
)


print('Total Artist-year profiles:', len(artist_year))
print('Total artists:', artist_year['track_artist'].nunique())
print('\nSample output:')
artist_year.head(10)

Total Artist-year profiles: 4568
Total artists: 1094

Sample output:


,track_artist,release_year,mean_popularity,median_popularity,max_popularity,track_count,mean_danceability,mean_energy,mean_valence,mean_tempo,mean_acousticness,mean_speechiness
0,$uicideBoy$,2015,62.7273,63.0,77,11,0.7530,0.5929,0.1967,114.1974,0.1842,0.1191
1,$uicideBoy$,2016,58.2857,64.0,68,7,0.8336,0.7509,0.2984,116.9920,0.1761,0.1046
2,$uicideBoy$,2017,57.0000,61.0,73,5,0.7858,0.7628,0.2992,127.0032,0.0569,0.1833
3,$uicideBoy$,2018,59.0000,59.0,62,2,0.6410,0.7115,0.2720,133.4470,0.2670,0.3685
4,*NSYNC,1997,54.0000,54.0,54,1,0.7580,0.9300,0.8530,112.0420,0.0503,0.0425
5,*NSYNC,2000,71.0000,71.0,71,1,0.6100,0.9260,0.8610,172.6380,0.0310,0.0479
6,*NSYNC,2005,43.0000,43.0,43,1,0.3750,0.5560,0.3980,111.8240,0.3980,0.0411
7,11 Acorn Lane,2014,22.0000,22.0,22,1,0.8420,0.8710,0.9490,119.9700,0.0587,0.0441
8,11 Acorn Lane,2016,43.0000,43.0,43,1,0.7930,0.8680,0.8500,124.9850,0.0304,0.1730
9,11 Acorn Lane,2018,40.0000,40.0,40,1,0.6720,0.9780,0.5670,126.0220,0.1380,0.1060


In [ ]:
# create features that help with predictions
# sort by artist name then year
artist_year = artist_year.sort_values(['track_artist', 'release_year'])
artist_year = artist_year.reset_index(drop=True)

#1 change over years: ow much did their popularity change from last year?
artist_year['yearly_change'] = artist_year.groupby('track_artist')['mean_popularity'].diff()

#2 variability: how much does this artists popularity jump around over 3 years?
# for each artist, calculate the 3 year standard deviation
variability = artist_year.groupby('track_artist')['mean_popularity'].rolling(window=3,min_periods=2).std()
variability = variability.reset_index(level=0, drop=True)
artist_year['variability'] = variability

#3 Career year: what year of their career is this?
#first year each artist appeared
first_year = artist_year.groupby('track_artist')['release_year'].transform('min')
artist_year['career_age'] = artist_year['release_year'] - first_year + 1

artist_year[['track_artist', 'release_year', 'mean_popularity',
             'yearly_change', 'variability', 'career_age']].head(12)

# peak popularity up to this point in career
artist_year['peak_popularity'] = artist_year.groupby('track_artist')['mean_popularity'].transform('max')

# how far below their peak they currently are
artist_year['gap_from_peak'] = artist_year['mean_popularity'] - artist_year['peak_popularity']

# how many years since they were at their peak
peak_year = artist_year.loc[
    artist_year.groupby('track_artist')['mean_popularity'].idxmax()
][['track_artist', 'release_year']].rename(columns={'release_year': 'peak_year'})
artist_year = artist_year.merge(peak_year, on='track_artist')
artist_year['years_since_peak'] = artist_year['release_year'] - artist_year['peak_year']
artist_year = artist_year.drop(columns=['peak_year'])

# overall career slope - average yearly change across career
artist_year['career_slope'] = artist_year.groupby('track_artist')['yearly_change'].transform('mean')

# did the artist release music the previous year
artist_year = artist_year.sort_values(['track_artist', 'release_year'])
artist_year['release_consistency'] = (
    artist_year.groupby('track_artist')['release_year'].diff() == 1
).astype(int)

## Findings — Features

yearly_change: how much mean popularity changed from the previous year.
Negative = popularity dropped, positive = popularity increased.
First year of each artist is NaN since there is no previous year.

variability: how unstable an artists popularity has been over 3 years.
Higher number means more unstable. 

career_age: what year of their career this row represents.
Year 1 = first active year, year 2 = second active year...

All three features capture how an artist is changing over time.

In [17]:
#define the decline Label

# get each artists popularity the following year
artist_year['next_popularity'] = artist_year.groupby('track_artist')['mean_popularity'].shift(-1)

# label as declined if next year drops 10 or more points
artist_year['declined'] = ((artist_year['next_popularity'] - artist_year['mean_popularity']) <= -10).astype(int)

# drop the last year of each artist because we dont know what happens after 
artist_year_labeled = artist_year.dropna(subset=['next_popularity']).copy()

print('Total rows:    ', len(artist_year_labeled))
print('Declined rows: ', artist_year_labeled['declined'].sum())
print('Decline rate:  ', round(artist_year_labeled['declined'].mean() * 100, 1), '%')


Total rows:     3474
Declined rows:  889
Decline rate:   25.6 %


## Findings — Decline Label

We defined decline as a drop of 10 or more popularity points from one year to the next. For each artist-year row we looked at what their popularity would be the following year and labeled it accordingly.

declined = 1 means the artist dropped 10+ points the next year
declined = 0 means the artist did not drop 10+ points the next year

Our results mean roughly 1 in 4 artist-years in our dataset shows a meaningful popularity decline the following year. This is a reasonable balance between the two classes and gives our classification models enough decline examples to learn from.

The last year of each artists data was dropped because we cannot know what happens after their final year, so those rows cannot be labeled.




In [19]:
# define 3 class trajectory label

def assign_trajectory(row):
    if pd.isna(row['next_popularity']):
        return None
    change = row['next_popularity'] - row['mean_popularity']
    # declining = drops 10+ points next year
    if change <= -10:
        return 'declining'
    # growing = gains 10+ points next year
    elif change >= 10:
        return 'growing'
    # stable = everything in between
    else:
        return 'stable'

artist_year_labeled['trajectory'] = artist_year_labeled.apply(assign_trajectory, axis=1)

print(artist_year_labeled['trajectory'].value_counts())

trajectory
stable       1522
growing      1063
declining     889
Name: count, dtype: int64


In [20]:
#Save final Dataset

notebook_dir = os.path.abspath('')
processed_dir = os.path.join(notebook_dir, '..', '..', 'data', 'processed')
os.makedirs(processed_dir, exist_ok=True)
output_file = os.path.join(processed_dir, 'artist_year_profiles.csv')
artist_year_labeled.to_csv(output_file, index=False)

print('File saved successfully')
print('Location:', output_file)
print('Rows saved:', len(artist_year_labeled))
print('Columns saved:', artist_year_labeled.columns.tolist())

File saved successfully
Location: /Users/kultumlhabaik/Documents/data-mining-final-project/notebooks/Data Preparation/../../data/processed/artist_year_profiles.csv
Rows saved: 3474
Columns saved: ['track_artist', 'release_year', 'mean_popularity', 'median_popularity', 'max_popularity', 'track_count', 'mean_danceability', 'mean_energy', 'mean_valence', 'mean_tempo', 'mean_acousticness', 'mean_speechiness', 'yearly_change', 'variability', 'career_age', 'peak_popularity', 'gap_from_peak', 'years_since_peak', 'career_slope', 'release_consistency', 'next_popularity', 'declined', 'trajectory']
